In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

In [3]:
month_map = {'April': 4, 'May': 5, 'June': 6}
portfolios = ['A', 'B', 'C', 'D']
all_interval = []

for p in portfolios:
    df = pd.read_excel('./data/Data for Datathon (Revised).xlsx', sheet_name=f"{p} - Interval")
    df['year'] = 2025
    df['month_num'] = df['Month'].map(month_map)
    df['datetime'] = pd.to_datetime(
        df['year'].astype(str) + '-' + df['month_num'].astype(str) + '-' + df['Day'].astype(str)
    )
    df['interval_time'] = df['Interval'].apply(
        lambda x: pd.Timedelta(hours=x.hour, minutes=x.minute) if pd.notnull(x) else pd.NaT
    )
    df['datetime'] = df['datetime'] + df['interval_time']
    df['portfolio'] = p
    all_interval.append(df)

combined = pd.concat(all_interval, ignore_index=True).sort_values(['portfolio', 'datetime']).reset_index(drop=True)
print(combined.shape)
combined.head()

(17102, 13)


,Month,Day,Interval,Service Level,Call Volume,Abandoned Calls,Abandoned Rate,CCT,year,month_num,datetime,interval_time,portfolio
0,April,1,00:00:00,1.0,5.0,0.0,0.0,137.60,2025,4,2025-04-01 00:00:00,0 days 00:00:00,A
1,April,1,00:30:00,1.0,5.0,0.0,0.0,263.40,2025,4,2025-04-01 00:30:00,0 days 00:30:00,A
2,April,1,01:00:00,1.0,4.0,0.0,0.0,333.25,2025,4,2025-04-01 01:00:00,0 days 01:00:00,A
3,April,1,01:30:00,1.0,3.0,0.0,0.0,170.00,2025,4,2025-04-01 01:30:00,0 days 01:30:00,A
4,April,1,02:00:00,1.0,1.0,0.0,0.0,667.00,2025,4,2025-04-01 02:00:00,0 days 02:00:00,A


In [4]:
daily_dfs = {}

for p in portfolios:
    df = pd.read_excel('./data/Data for Datathon (Revised).xlsx', sheet_name=f"{p} - Daily")
    df['date'] = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df = df.rename(columns={
        'Call Volume': 'daily_cv',
        'CCT': 'daily_cct',
        'Abandon Rate': 'daily_abd'
    })
    df['portfolio'] = p
    daily_dfs[p] = df[['date', 'daily_cv', 'daily_cct', 'daily_abd', 'portfolio']]

daily_all = pd.concat(daily_dfs.values(), ignore_index=True)
print(daily_all.shape)

(2924, 5)


In [5]:
# Core time features
combined['interval_of_day'] = combined['datetime'].dt.hour * 2 + (combined['datetime'].dt.minute // 30)
combined['day_of_week'] = combined['datetime'].dt.dayofweek  # 0=Mon, 6=Sun
combined['day_of_month'] = combined['datetime'].dt.day
combined['month'] = combined['datetime'].dt.month
combined['week_of_month'] = (combined['day_of_month'] - 1) // 7 + 1
combined['is_weekend'] = (combined['day_of_week'] >= 5).astype(int)

# Cyclical encoding so the model understands 23:30 and 00:00 are neighbors
combined['hour_sin'] = np.sin(2 * np.pi * combined['interval_of_day'] / 48)
combined['hour_cos'] = np.cos(2 * np.pi * combined['interval_of_day'] / 48)
combined['dow_sin'] = np.sin(2 * np.pi * combined['day_of_week'] / 7)
combined['dow_cos'] = np.cos(2 * np.pi * combined['day_of_week'] / 7)

# Holidays in our training window
holidays = pd.to_datetime(['2025-05-26', '2025-06-19'])  # Memorial Day, Juneteenth
combined['is_holiday'] = combined['datetime'].dt.normalize().isin(holidays).astype(int)

# Portfolio as a number
combined['portfolio_enc'] = combined['portfolio'].map({'A': 0, 'B': 1, 'C': 2, 'D': 3})

# Merge daily totals in as a feature (gives model the "daily scale" context)
combined['date'] = combined['datetime'].dt.normalize()
daily_all = pd.concat(daily_dfs.values(), ignore_index=True)
combined = combined.merge(daily_all, on=['date', 'portfolio'], how='left')

print(combined[['interval_of_day', 'day_of_week', 'is_weekend', 'is_holiday', 
                 'hour_sin', 'hour_cos', 'portfolio_enc', 'daily_cv']].head(10))

   interval_of_day  day_of_week  is_weekend  is_holiday  hour_sin  hour_cos  \
0              0.0          1.0           0           0  0.000000  1.000000   
1              1.0          1.0           0           0  0.130526  0.991445   
2              2.0          1.0           0           0  0.258819  0.965926   
3              3.0          1.0           0           0  0.382683  0.923880   
4              4.0          1.0           0           0  0.500000  0.866025   
5              5.0          1.0           0           0  0.608761  0.793353   
6              6.0          1.0           0           0  0.707107  0.707107   
7              7.0          1.0           0           0  0.793353  0.608761   
8             10.0          1.0           0           0  0.965926  0.258819   
9             11.0          1.0           0           0  0.991445  0.130526   

   portfolio_enc  daily_cv  
0              0    5249.0  
1              0    5249.0  
2              0    5249.0  
3             